In [3]:
# import library
import pandas as pd

In [5]:
# import dataset
file_path = "Group 8- original dataset-Electronic_sales_Sep2023-Sep2024.csv"
sale_data = pd.read_csv(file_path)

## PART 1: CLEANING

In [6]:
# data preprocessing & cleaning
# step 1:transfer date form
sale_data["Purchase Date"] = pd.to_datetime(sale_data["Purchase Date"], errors="coerce")

# step 2:delete "cancelled" orders
#boolean indexing, filtering completed orders only
is_completed = sale_data["Order Status"] == "Completed"

# apply the filter
sales_cleaned = sale_data[is_completed].copy()

#step 3:replacing the mode to "N/A" cell in gender field
#we use [0] to get the actual value, as .mode() returns pandas series
gender_mode= sales_cleaned["Gender"].mode()[0]

sales_cleaned["Gender"]= sales_cleaned["Gender"].replace("#N/A",gender_mode).fillna(gender_mode)

#step 4: payment method buckets
payment_bucket= {
    'Debit Card': 'Card',
    'Credit Card': 'Card',
    'PayPal': 'Online Payment',
    'Paypal': 'Online Payment',
    'Bank Transfer': 'Online Payment',
    'Cash': 'Cash'
}
#apply mapping to create new column
sales_cleaned['payment_classification'] = sales_cleaned['Payment Method'].map(payment_bucket)

## PART 2: RFM AGGREGATION

In [7]:
# create a new set to caculate
rfm_data = []

# get the unique customer id
customers = sales_cleaned["Customer ID"].unique()

# calculate each customer's R, F, M; identify the calculation formulas as below:
## R = last snapshot_date - last purchase date, while snapshot date was defined later;
## F = purchase times (count customer ID);
## M = total purchase cost (sum of total price of each customer).

#set a snapshot date = the last date of purchase(all customers) + 1
snapshot_date = sales_cleaned["Purchase Date"].max() + pd.Timedelta(days=1)

for cid in customers:
    customer_data = sales_cleaned[sales_cleaned["Customer ID"] == cid]
    
    # calculate R
    first_date = customer_data["Purchase Date"].min()
    last_date = customer_data["Purchase Date"].max()
    R = (snapshot_date - last_date).days

    # calculate F
    F = len(customer_data)
    
    # calculate M
    M = customer_data["Total Price"].sum()
        
    # calculate lisespan (for later customer score)
    lifespan = (last_date - first_date).days + 1
    # calculate average_purchase_value (for later customer score)
    average_purchase_value = M / customer_data["Customer ID"].count()
    
    #aggregated payment method string
    #need the payment_classification to be etc customer 1000: "cash, card, card"
    agg_payment_string = ", ".join(customer_data["payment_classification"].astype(str))
    
    # put the results into rfm_data
    rfm_data.append([cid, R, F, M, lifespan, average_purchase_value, agg_payment_string])

## PART 3: RFM SCORING

In [8]:
# transfer to pandas DataFrame
customer_analysis = pd.DataFrame(rfm_data, columns=["Customer ID", "R", "F", "M", "lifespan", "average_purchase_value","agg_payment_class"])

# set score methods for R, F, M, the rules are identified as below (compare each single customer with population):
## if R_single < average R_population, assign it with 1, otherwise assign with 0;
## if F_single > average F_population, assign it with 1, otherwise assign with 0;
## if M_single > median M_population, assign it with 1, otherwise assign with 0:
### the reason why we choose median_M instead of average_M is for better and more significant categorizing

# calculate the average/median threshold
R_mean = customer_analysis["R"].mean()
F_mean = customer_analysis["F"].mean()
M_median = customer_analysis["M"].median()

# score for each customer
R_score_list = []
F_score_list = []
M_score_list = []
RFM_value_list = []

for i in range(len(customer_analysis)):
    r = customer_analysis.loc[i, "R"]
    f = customer_analysis.loc[i, "F"]
    m = customer_analysis.loc[i, "M"]
    
    # score R
    if r < R_mean:
        R_score = 1
    else:
        R_score = 0
    
    # score F
    if f > F_mean:
        F_score = 1
    else:
        F_score = 0
    
    # socre M
    if m > M_median:
        M_score = 1
    else:
        M_score = 0
    
    # joint R, F, M
    RFM_value = str(R_score) + str(F_score) + str(M_score)
    
    # store the results
    R_score_list.append(R_score)
    F_score_list.append(F_score)
    M_score_list.append(M_score)
    RFM_value_list.append(RFM_value)

# add results to final dataset
customer_analysis["R_score"] = R_score_list
customer_analysis["F_score"] = F_score_list
customer_analysis["M_score"] = M_score_list
customer_analysis["RFM_value"] = RFM_value_list

print(customer_analysis.head())

   Customer ID    R  F         M  lifespan  average_purchase_value  \
0         1000  157  1    741.09         1                 741.090   
1         1002   46  2   5020.60       298                2510.300   
2         1003  126  1     41.50         1                  41.500   
3         1004  121  1     83.00         1                  83.000   
4         1005   92  2  11779.11       147                5889.555   

      agg_payment_class  R_score  F_score  M_score RFM_value  
0        Online Payment        0        0        0       000  
1            Card, Cash        1        1        1       111  
2                  Cash        1        0        0       100  
3                  Card        1        0        0       100  
4  Online Payment, Card        1        1        1       111  


In [28]:
# customer score
# we want to socre customers based on thier shopping behaviours, higher socre, better customer

customer_score_list = []

for i in range(len(customer_analysis)):
    lifespan = customer_analysis.loc[i, "lifespan"]
    frequency = customer_analysis.loc[i, "F"]
    average_purchase_value = customer_analysis.loc[i, "average_purchase_value"]
    
    #calculate customer score
    customer_score = lifespan * frequency * average_purchase_value
    # store the results
    customer_score_list.append(customer_score)
    
# add results to final dataset
customer_analysis["customer_score"] = customer_score_list

customer_analysis.head()

,Customer ID,R,F,M,lifespan,average_purchase_value,agg_payment_class,R_score,F_score,M_score,RFM_value,customer_score
0,1000,157,1,741.09,1,741.090,Online Payment,0,0,0,000,741.09
1,1002,46,2,5020.60,298,2510.300,"Card, Cash",1,1,1,111,1496138.80
2,1003,126,1,41.50,1,41.500,Cash,1,0,0,100,41.50
3,1004,121,1,83.00,1,83.000,Card,1,0,0,100,83.00
4,1005,92,2,11779.11,147,5889.555,"Online Payment, Card",1,1,1,111,1731529.17


## PART 4: CUSTOMER BUCKET AND PRECISE MARKETING

In [9]:
def create_rfm_bucket(df):
    Customer_Segment_list = []
    Precise_Marketing_list = []

    for i in range(len(df)):
        R_score = df.loc[i, "R_score"]
        F_score = df.loc[i, "F_score"]
        M_score = df.loc[i, "M_score"]

        RFM_value = f"{R_score}{F_score}{M_score}"

        if RFM_value == "111":
            Customer_Segment = "Champion"
            Precise_Marketing = ("Give special rewards and priority access to new products.")
    
        elif RFM_value == "110":
            Customer_Segment = "Loyal Customer"
            Precise_Marketing = ("Increase engagement with exclusive offers by conducting personal email campaigns and referral programs.")
    
        elif RFM_value == "101":
            Customer_Segment = "Potential Loyalist"
            Precise_Marketing = ("Convert them into loyal customers by providing discounts on second purchases.")
    
        elif RFM_value == "100":
            Customer_Segment = "Promisisng Customer"
            Precise_Marketing = ("Encourage them to increase purchases through special promos for second purchases, upselling, and cross-selling.")
    
        elif RFM_value in ["011", "010"]:
            Customer_Segment = "Needs Attention"
            Precise_Marketing = ("Bring back their interest by sending limited-time offers.")
    
        elif RFM_value == "001":
            Customer_Segment = "At Risk"
            Precise_Marketing = ("Identify the reasons they stop shopping with satisfaction surveys and return special offers.")
    
        else:  # 000
            Customer_Segment = "Can't Lose Them"
            Precise_Marketing = ("Reactivate with attractive offers by carrying out big discounts, bundling offers, and reminder campaigns via email." )

        Customer_Segment_list.append(Customer_Segment)
        Precise_Marketing_list.append(Precise_Marketing)

    df["Customer_Segment"] = Customer_Segment_list
    df["Precise_Marketing"] = Precise_Marketing_list

    return df

In [10]:
customer_analysis = create_rfm_bucket(customer_analysis)

In [11]:
customer_analysis.head()

,Customer ID,R,F,M,lifespan,average_purchase_value,agg_payment_class,R_score,F_score,M_score,RFM_value,Customer_Segment,Precise_Marketing
0,1000,157,1,741.09,1,741.090,Online Payment,0,0,0,000,Can't Lose Them,Reactivate with attractive offers by carrying ...
1,1002,46,2,5020.60,298,2510.300,"Card, Cash",1,1,1,111,Champion,Give special rewards and priority access to ne...
2,1003,126,1,41.50,1,41.500,Cash,1,0,0,100,Promisisng Customer,Encourage them to increase purchases through s...
3,1004,121,1,83.00,1,83.000,Card,1,0,0,100,Promisisng Customer,Encourage them to increase purchases through s...
4,1005,92,2,11779.11,147,5889.555,"Online Payment, Card",1,1,1,111,Champion,Give special rewards and priority access to ne...


## PART 5: CUSTOMER PAYMENT PRIORITY

In [12]:
def create_payment_priority(row):
    #extract from row(1 row=1 customer)
    #take from field F to variable F
    F=row['F']
    agg_payments = row['agg_payment_class']

    #CASE 1:only 1 order
    if F==1:
        return "Sample too small (F=1)"

    #Count occurence of each payment method type
    typem_count= pd.Series(agg_payments.split(', ')).value_counts()

    #CASE 2: >=2 orders but each order is unique methods
    if len(typem_count)>1 and typem_count.iloc[0] == typem_count.iloc[1]:
        return "No Priority"

    #CASE 3: >=2 orders and some methods used are repeated
    if F>1 and typem_count.iloc[0]> typem_count.iloc[1] if len(typem_count) >1 else True:
        majority_method=typem_count.index[0]
        return f"Mainly by {majority_method}"
    return "Classification Error"

customer_analysis['payment_priority'] = customer_analysis.apply(create_payment_priority, axis=1)

print(customer_analysis[['F','agg_payment_class','payment_priority']].head())

   F     agg_payment_class        payment_priority
0  1        Online Payment  Sample too small (F=1)
1  2            Card, Cash             No Priority
2  1                  Cash  Sample too small (F=1)
3  1                  Card  Sample too small (F=1)
4  2  Online Payment, Card             No Priority


## PART 6: CUMSOMER MEMBERSHIP TYPE

In [13]:
# identify the final customer membership type of each customer
# Types of memership: New / Regular / Churned / Non
# 1. New Member      → A customer who was previously not a member ("No") 
#                       but later became a member ("Yes").
#
# 2. Regular Member  → A customer who is consistently a member ("Yes")
#                       across their purchase history.
#
# 3. Churned Member  → A customer who used to be a member ("Yes") 
#                       but later stopped being a member ("No").
#
# 4. Non Member      → A customer who never joined the membership program
#                       ("No" in all purchase records) or has no membership data.

#set purchase date
sale_data['Purchase Date'] = pd.to_datetime(sale_data['Purchase Date'])
df_sorted = sale_data.sort_values(['Customer ID', 'Purchase Date'])

#set a list to identify and store customer membership type during traversals
membership_records = []

# identify changes of each customer
for customer_id, group in df_sorted.groupby('Customer ID'):
    previous_status = None
    last_type = 'Non Member'  # set "Non Member" as original status
    
    for _, row in group.iterrows():
        current_status = row['Loyalty Member']
        
        if previous_status is None:
            if current_status == 'Yes':
                last_type = 'Regular Member'
            else:
                last_type = 'Non Member'
        else:
            if previous_status == 'Yes' and current_status == 'No':
                last_type = 'Churned'
            elif previous_status == 'No' and current_status == 'Yes':
                last_type = 'New Member'
            elif current_status == 'Yes':
                last_type = 'Regular Member'
            else:
                last_type = 'Non Member'
        previous_status = current_status

    membership_records.append({'Customer ID': customer_id, 'Membership Type': last_type})

# transfer the final status to DataFrame
membership_df = pd.DataFrame(membership_records)

# add to customer_analysis
customer_analysis = customer_analysis.merge(membership_df, on='Customer ID', how='left')

# input "Non Member" to customers with no membership data
customer_analysis['Membership Type'] = customer_analysis['Membership Type'].fillna('Non Member')

In [14]:
customer_analysis.head()

,Customer ID,R,F,M,lifespan,average_purchase_value,agg_payment_class,R_score,F_score,M_score,RFM_value,Customer_Segment,Precise_Marketing,payment_priority,Membership Type
0,1000,157,1,741.09,1,741.090,Online Payment,0,0,0,000,Can't Lose Them,Reactivate with attractive offers by carrying ...,Sample too small (F=1),Non Member
1,1002,46,2,5020.60,298,2510.300,"Card, Cash",1,1,1,111,Champion,Give special rewards and priority access to ne...,No Priority,New Member
2,1003,126,1,41.50,1,41.500,Cash,1,0,0,100,Promisisng Customer,Encourage them to increase purchases through s...,Sample too small (F=1),Regular Member
3,1004,121,1,83.00,1,83.000,Card,1,0,0,100,Promisisng Customer,Encourage them to increase purchases through s...,Sample too small (F=1),Non Member
4,1005,92,2,11779.11,147,5889.555,"Online Payment, Card",1,1,1,111,Champion,Give special rewards and priority access to ne...,No Priority,Non Member


In [15]:
#final output csv
output_file_path = 'sales_data_analysisoutput.csv'
customer_analysis.to_csv(output_file_path, index=False) # index=False prevents writing the DataFrame index as a column